In [ ]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import joblib
import warnings
import time

warnings.filterwarnings("ignore")

from google.colab import drive
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor


# ==========================================
# 2. MOUNT GOOGLE DRIVE
# ==========================================
drive.mount('/content/drive')

# ==========================================
# 3. LOAD DATA
# ==========================================
csv_file_path = '/content/drive/MyDrive/Colab Notebooks/Công nghệ IoT nâng cao/final_data.csv'
df = pd.read_csv(csv_file_path)

df = df.drop(columns=["Unnamed: 0","label"], errors="ignore")

print("Original shape:", df.shape)



df["hour"] = df["datetime"].dt.hour
df["minute"] = df["datetime"].dt.minute
df["day"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["minute_sin"] = np.sin(2 * np.pi * df["minute"] / 60)
df["minute_cos"] = np.cos(2 * np.pi * df["minute"] / 60)

df = df.drop(columns=["datetime"])

print("After feature engineering:", df.shape)

# ==========================================
# 5. CREATE LAG FEATURES
# ==========================================
def create_features(df, lag=10):

    X, y = [], []
    values = df.values

    for i in range(lag, len(values)):
        X.append(values[i-lag:i].flatten())
        y.append(values[i][0:3])  # temp, humidity, pressure

    return np.array(X), np.array(y)

lag = 10
X, y = create_features(df, lag)

print("X shape:", X.shape)
print("y shape:", y.shape)

# ==========================================
# 6. TRAIN / TEST SPLIT
# ==========================================
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ==========================================
# 7. TRAIN MODELS + TIME MEASURE
# ==========================================
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": MultiOutputRegressor(
        GradientBoostingRegressor(random_state=42)
    )
}

results = []

total_start = time.time()

for name, model in models.items():

    print(f"\nTraining {name}...")

    # ⏱ Train time
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    # ⏱ Predict time
    start_pred = time.time()
    pred = model.predict(X_test)
    pred_time = time.time() - start_pred

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    print(f"Train time: {train_time:.3f} sec")
    print(f"Predict time: {pred_time:.3f} sec")

    results.append([name, mae, rmse, train_time, pred_time])

total_time = time.time() - total_start

# ==========================================
# 8. COMPARE RESULTS
# ==========================================
results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "Train Time (s)", "Predict Time (s)"]
)

results_df = results_df.sort_values("RMSE")

print("\n===== MODEL COMPARISON =====")
print(results_df)

print(f"\nTotal training time: {total_time:.2f} sec")

best_model_name = results_df.iloc[0]["Model"]
print("\nBest model:", best_model_name)

# ==========================================
# 9. SAVE BEST MODEL
# ==========================================
best_model = models[best_model_name]

joblib.dump({
    "model": best_model,
    "lag": lag,
    "feature_count": df.shape[1],
    "target_count": 3
}, "best_weather_model.pkl")

print("\nModel saved as best_weather_model.pkl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original shape: (48233, 3)
After feature engineering: (48233, 11)
X shape: (48223, 110)
y shape: (48223, 3)

Training LinearRegression...
Train time: 0.543 sec
Predict time: 0.007 sec

Training RandomForest...
Train time: 464.334 sec
Predict time: 0.304 sec

Training GradientBoosting...
Train time: 194.938 sec
Predict time: 0.035 sec

===== MODEL COMPARISON =====
              Model       MAE      RMSE  Train Time (s)  Predict Time (s)
0  LinearRegression  0.139413  0.413441        0.542842          0.007211
2  GradientBoosting  0.177791  0.513645      194.937721          0.035263
1      RandomForest  0.550994  0.887045      464.334152          0.303949

Total training time: 660.17 sec

Best model: LinearRegression

Model saved as best_weather_model.pkl


In [ ]:
y_train

array([31.64, 31.66, 31.68, ..., 28.37, 28.35, 28.34])

In [ ]:
df.tail(5)

,temperature,humidity,pressure,hour,minute,day,dayofweek,month,hour_sin,hour_cos,minute_sin,minute_cos
48228,31.22,61.10,1010.25,11,48,3,5,2,0.258819,-0.965926,-0.951057,0.309017
48229,31.19,61.12,1010.25,11,49,3,5,2,0.258819,-0.965926,-0.913545,0.406737
48230,31.20,61.29,1010.23,11,50,3,5,2,0.258819,-0.965926,-0.866025,0.500000
48231,31.23,61.19,1010.27,11,51,3,5,2,0.258819,-0.965926,-0.809017,0.587785
48232,31.15,61.42,1010.26,11,52,3,5,2,0.258819,-0.965926,-0.743145,0.669131


In [ ]:
df.head(5)

,temperature,humidity,pressure,hour,minute,day,dayofweek,month,hour_sin,hour_cos,minute_sin,minute_cos
0,31.58,62.18,1012.87,0,0,1,0,1,0.0,1.0,0.000000,1.000000
1,31.59,62.27,1012.89,0,1,1,0,1,0.0,1.0,0.104528,0.994522
2,31.59,62.96,1012.88,0,2,1,0,1,0.0,1.0,0.207912,0.978148
3,31.61,62.92,1012.89,0,3,1,0,1,0.0,1.0,0.309017,0.951057
4,31.59,63.02,1012.89,0,4,1,0,1,0.0,1.0,0.406737,0.913545
